# Izdajem Iznajmljujem - LLM Chatbot za korisnicku podrsku

Cilj ovog sistema je da pruzi korisnicima platforme automatsku podrsku zasnovanu na pravilima koriscenja. Umesto statickih odgovora, chatbot razume nameru korisnika, pronalazi relevantna pravila iz vektorske baze i generise precizan odgovor u realnom vremenu.

Sistem je implementiran u 5 faza:

1. **Priprema okruzenja** (Instalacija zavisnosti i autentifikacija)
2. **Baza znanja i RAG pipeline** (Chunking, vektorizacija i pretraga)
3. **LLM i prompt inzenjering** (GPT-4o-mini sa sistemskim kontekstom)
4. **Agentic arhitektura** (LangGraph multi-node graf sa memorijom i uslovnim grananjem)
5. **Testiranje** (Scenariji za sve grane grafa)

## Faza 1: Priprema okruženja

Instaliramo neophodne biblioteke i postavljamo OpenAI API kljuc koji se koristi za embeddings i chat model.

- **LangChain** - okvir za izgradnju LLM aplikacija
- **langchain-openai** - integracija sa OpenAI (GPT-4o-mini i text-embedding-3-small)
- **langchain-chroma** - konektor za Chroma vektorsku bazu
- **tiktoken** - tokenizator koji OpenAI modeli koriste interno za brojanje tokena

In [ ]:
!pip install -q langchain langchain-openai langchain-chroma tiktoken langchain-community


In [ ]:
 os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

NameError: name 'userdata' is not defined

## Faza 2: Baza znanja i RAG pipeline

### Baza znanja - pravila platforme

Baza znanja sadrzi 18 pravila platforme koja pokrivaju: postavljanje oglasa, promotivne pakete, dopunu kredita, depozite, odgovornost pri ostecenju, otkazivanje ugovora, zabranjene oglase i GDPR prava korisnika.

Pravila se cuavaju u tekstualnom fajlu `pravila_izdavanja.txt` koji se zatim ucitava u LangChain `TextLoader`.

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
platform_rules = """
1. Pravo na postavljanje oglasa imaju isključivo lica koja su punoletna (18+ godina) i koja su prošla email verifikaciju.
2. Korisnici koji su prošli identifikaciju (KYC — upload ličnog dokumenta) dobijaju oznaku "Verifikovan" na profilu i imaju veće poverenje među drugim korisnicima.
3. Postavljanje oglasa je besplatno. Promotivne opcije se naplaćuju u kreditima: FEATURED 500 RSD (7 dana, vrh pretrage), PRIORITY 250 RSD (3 dana), HIGHLIGHTED 100 RSD (30 dana, vizuelni badge).
4. Dopuna kredita vrši se uplatom na zvanični račun platforme (Dimitrije Mitic, 265-0000006785327-58) uz obavezno korišćenje jedinstvenog poziva na broj koji se generiše na stranici `/credit`.
5. Krediti nisu povratni nakon što su iskorišćeni za aktivaciju promocije. Ako promocija nije konzumirana, korisnik može tražiti povraćaj u roku od 7 dana na mejl podrške.
6. Depozit koji zakupac eventualno uplaćuje zakupodavcu obavezno se vraća po isteku ugovora, ukoliko predmet iznajmljivanja nije oštećen i ukoliko je vraćen u dogovorenom roku.
7. Stanje iznajmljene stvari (fotografije, funkcionalnost) obavezno se dokumentuje pri primopredaji — i preporučuje se razmena tih fotografija u chat-u platforme kao dokaz.
8. Ukoliko stanodavac (vlasnik stvari) otkaže već prihvaćen ugovor bez opravdanog razloga, zakupac ima pravo na prioritetnu podršku i eventualnu naknadu troškova dokazanih kroz chat istoriju.
9. Otkazni rok za već prihvaćen ugovor je minimum 48 sati pre datuma početka — kasnije otkazivanje može rezultirati negativnom recenzijom i suspenzijom naloga kod ponavljanja.
10. Oštećenje iznajmljene stvari mora biti prijavljeno u roku od 48 sati od primopredaje. Prijava se vrši kroz chat platforme i mejlom na izdajemiznajmljujem.rs@gmail.com sa fotografijama.
11. Platforma ne posreduje u plaćanju naknade za rental (iznajmljivanje) — to se dogovara direktno između korisnika. Platforma naplaćuje isključivo promotivne opcije putem sistema kredita.
12. Zabranjeno je oglašavanje: oružja, narkotika, životinja, ljudskih organa, krađene robe, falsifikata, pirotehnike, alkohola licima mlađim od 18 godina, kao i bilo kakvih nelegalnih usluga.
13. Korisnik ima pravo na brisanje naloga i svih ličnih podataka u skladu sa Zakonom o zaštiti podataka o ličnosti (ZZPL) i GDPR-om — zahtev se šalje na izdajemiznajmljujem.rs@gmail.com.
14. Kasno vraćanje iznajmljene stvari može rezultirati dodatnom naknadom za svaki započet dan kašnjenja prema ceni iz originalnog oglasa.
15. Sporovi oko ugovora rešavaju se najpre direktno kroz chat platforme. Ukoliko korisnici ne postignu dogovor, slučaj se eskaliraju na izdajemiznajmljujem.rs@gmail.com — admini posreduju u roku od 3 radna dana.
16. Prijava sumnje na prevaru (traženje uplate na privatne račune van /credit stranice, lažni oglasi, pretnje) mora biti bezuslovna — suspenzija naloga je trenutna, a slučaj se predaje nadležnim organima ako postoje elementi krivičnog dela.
17. Recenzije se mogu ostaviti isključivo nakon završenog ugovora, u roku od 30 dana. Lažne recenzije, međusobno "nameštanje" i vređanja se brišu bez upozorenja.
18. Prava potrošača u skladu sa Zakonom o zaštiti potrošača — pravo na reklamaciju neispravne usluge, pravo na informisanost, pravo na zaštitu od nepoštene poslovne prakse — primenjuju se na sve komercijalne transakcije preko platforme.
"""

In [ ]:
with open("pravila_izdavanja.txt", "w", encoding = "utf-8") as f:
  f.write(platform_rules)

In [ ]:
loader = TextLoader("pravila_izdavanja.txt")
document = loader.load()
print("Successfuly loaded file")

### Deljenje teksta - Chunking

Dugi tekst pravila se deli na manje komade jer jezicki modeli ne mogu efikasno da pretrazuju ceo dokument odjednom. `RecursiveCharacterTextSplitter` pokusava da podeli tekst na semanticki smislene celine - najpre po paragrafima, zatim po recenicama, i na kraju po karakterima ako je potrebno.

Parametri:
- `chunk_size=150` - maksimalni broj karaktera po komadu
- `chunk_overlap=20` - preklapanje izmedju susednih komada, tako da nijedno pravilo ne bude prerezano tacno na granici

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20
)
splitted_docs = text_splitter.split_documents(document)
print(f"Splitted on {len(splitted_docs)} chunks")

### Vektorizacija i vektorska baza - Chroma

Svaki komad teksta se pretvara u numericki vektor (embedding) koriscenjem OpenAI modela `text-embedding-3-small`. Vektori se cuvaju u **Chroma** bazi u memoriji, sto omogucava brzo pronalazenje semanticki slicnih komada na osnovu kosinusne slicnosti.

**Retriever** se postavlja sa `k=2` - pri svakom upitu iz Chroma baze vraca se 2 najslicnija komada teksta koji se zatim prosledjuju LLM-u kao kontekst.

In [ ]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectors = Chroma.from_documents(
    documents = splitted_docs,
    embedding=embeddings_model
)

In [ ]:
retriever = vectors.as_retriever(search_kwargs={"k": 2})

## Faza 3: LLM i prompt inzenjering

### Jezicki model i prompt sablon

Koristimo **GPT-4o-mini** kao jezicki model (`temperature=0.6` - umeren balans izmedju kreativnosti i preciznosti). Model prima kontekst iz vektorske baze i pitanje korisnika kroz `PromptTemplate` koji definise ulogu asistenta i instrukcije za ponasanje.

Osnovni RAG lanac (`rag_chain`) funkcionise na sledeci nacin: pitanje se konvertuje u embedding, vrsi se pretraga Chroma baze, pronadjeni komadi se umecu u prompt i LLM generise odgovor. Ovaj jednostavni lanac sluzi kao osnova na kojoj se gradi agentic arhitektura u fazi 4.

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

prompt_template = PromptTemplate(
    template = """Ti si gotivni ljubazni dobri agent za korisničku podršku sajta 'Izdajem Iznajmljujem'.
Na osnovu ponuđenog KONTEKSTA, odgovori na PITANJE korisnika.
Ako u KONTEKSTU nema odgovora, reci: "Nažalost, nemam tu informaciju u pravilniku." Nemoj nikada da izmišljaš pravila.

KONTEKST:
{context}

PITANJE KORISNIKA: {question}

ODGOVOR:""",
    input_variables=["context", "question"]
)

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

## Faza 4: Agentic arhitektura - LangGraph

### Graf i tok obrade

Umesto linearnog RAG lanca, koristimo **LangGraph** koji omogucava uslovnu logiku, grananje i pamcenje razgovora. Graf se sastoji od cvorova koji redom obraduju pitanje, a usmeravanje izmedju njih zavisi od sadrzaja stanja.

Tok obrade jedne poruke:

| Korak | Cvor | Uloga |
|-------|------|-------|
| 1 | `router` | Klasifikuje pitanje: platforma ili opsti razgovor |
| 2 | `retrieve_node` | Pronalazi relevantne komade iz Chroma baze |
| 3 | `grade_node` | Procenjuje da li komadi odgovaraju na pitanje |
| 4 | `generate_node` | Generise odgovor na osnovu konteksta |
| 4* | `escalate_node` | Fallback ako nema relevantnih podataka |
| 4* | `chat_node` | Direktan odgovor za pitanja van domene platforme |

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import operator


In [ ]:
# question = "Da li se placa  za oglas na vašem sajtu?"
# print(f"Korisnik pita: {question}")

# answer = rag_chain.invoke(question)
# print(f"Bot odgovara:\n{answer}")

### AgentState - zajednicko stanje grafa

Svaki cvor u LangGraph grafu prima trenutno stanje, obraduje ga i prosledjuje sledecem cvoru. `AgentState` definise koje informacije putuju kroz ceo tok obrade.

Polja koja prenosimo:
- **`question`** - originalno pitanje korisnika
- **`documents`** - komadi teksta pronadjeni u Chroma bazi
- **`expanded_queries`** - prosirene varijante pitanja (za bolju pokrivenost pretrage)
- **`chat_history`** - istorija razgovora (za pamcenje konteksta)
- **`is_relevant`** - ocena relevantnosti pronadjenih dokumenata (`yes` / `no`)
- **`answer`** - finalni odgovor koji se vraca korisniku

Polje `chat_history` koristi `operator.add` kao anotaciju, sto znaci da se nova istorija nadovezuje na postojecu umesto da je zamenjuje.

In [ ]:
class AgentState(TypedDict):
    question: str
    documents: List[str]
    expanded_queries: List[str]
    chat_history: Annotated[List[str], operator.add]
    is_relevant: str
    answer: str

### `query_expansion_node` - prosirenje upita

Pre pretrage vektorske baze, LLM preformulise originalno pitanje u 3 razlicite varijante. Ovim se povecava verovatnoca da ce Chroma pronaci relevantne komade cak i kada korisnik postavi pitanje drugacije od formulacije u pravilniku.

Na primer, za pitanje *"koliko kosta oglas"* LLM generise i: *"Da li postoji naknada za postavljanje oglasa?"* i *"Da li se oglasavanje naplacuje?"*. Originalno pitanje se takodje dodaje u listu, kao sigurnosna mreza u slucaju da su prosirene verzije previse daleke od originala.

In [ ]:
def query_expansion_node(state):
    question = state["question"]

    prompt = f"""Preformuliši sledeće korisničko pitanje u 3 različita, detaljnija pitanja na srpskom jeziku, kako bi olakšao pretragu pravila i dokumenata u bazi.
    Odgovori ISKLJUČIVO spiskom, svaki upit u novom redu, bez brojeva i bez dodatnog teksta.
    Originalno pitanje: {question}"""

    # LLM generiše 3 nova pitanja
    response = llm.invoke(prompt).content.strip()

    # Sečemo odgovor u listu stringova
    queries = response.split('\n')

    # Obavezno dodajemo i originalno pitanje u listu, zlu ne trebalo
    queries.append(question)

    return {"expanded_queries": queries}

### `retrieve_node` - pretraga vektorske baze

Cvor zaduzen za pronalazenje relevantnih informacija iz Chroma baze. Posto korisnici cesto postavljaju kratka i neprecizna pitanja, cvor najpre prosiruje upit pozivom LLM-a - generise 3 alternativne formulacije i dodaje originalno pitanje, ukupno 4 upita.

Za svaki upit vrsi se pretraga Chroma baze (k=2). Duplikati se filtriraju skupom `seen_content`, a u sledeci cvor se prosledjuje najvise 4 jedinstvena komada teksta. Uzima se u obzir i poslednjih 5 poruka iz istorije razgovora kako bi prosireni upiti bili relevantni za ceo tok konverzacije, a ne samo za poslednju poruku.

In [ ]:
def retrieve_node(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history", [])[-5:])

  expansion_prompt = f"""Korisnik je postavio kratko pitanje: '{question}'.
    Prethodni razgovor: {history}
    Tvoj zadatak je da preformulišeš ovo pitanje na 3 različita načina (sinonimi, profesionalniji rečnik), kako bismo što lakše pronašli odgovor u bazi pravila o iznajmljivanju.
    Vrati ISKLJUČIVO ta 3 pitanja, svako u novom redu, bez ikakvog dodatnog teksta, rednih brojeva ili znakova."""


  expanded_queries_text = llm.invoke(expansion_prompt).content
  expanded_queries = expanded_queries_text.strip().split('\n')

  expanded_queries.append(question)
  print(f"-> Prošireni upiti za pretragu:\n{expanded_queries_text}")

  all_docs = []
  seen_content = set()

  for q in expanded_queries:
    if not q.strip(): continue
    docs = retriever.invoke(q)
    for d in docs:
      if d.page_content not in seen_content:
        seen_content.add(d.page_content)
        all_docs.append(d.page_content)

  return {"documents": all_docs[:4]}

### `grade_node` - evaluacija relevantnosti (Self-RAG)

Ovaj cvor implementira koncept **Self-RAG** - model sam procenjuje da li su pronadjeni dokumenti zaista korisni za odgovor na pitanje. LLM dobija pitanje i pronadjeni kontekst i odgovara iskljucivo sa `yes` ili `no`.

Ovo sprecava situaciju u kojoj bi sistem generisao odgovor na osnovu irelevantnih komada teksta, sto bi dovelo do netacnih ili izmisljenih pravila (hallucination). Ocena se cita u `check_relevance` funkciji koja usmerava tok grafa.

In [ ]:
def grade_node(state):
  question = state["question"]
  context = "\n".join(state["documents"])

  prompt = f"""Proveri da li TEKST sadrži odgovor na PITANJE.
    Odgovori ISKLJUČIVO sa: yes ili no.
    TEKST: {context}
    PITANJE: {question}"""

  decision = llm.invoke(prompt).content.strip().lower()

  if "yes" in decision:
    return {"is_relevant": "yes"}
  else:
    return {"is_relevant": "no"}

### `generate_node` - generisanje odgovora

Cvor koji sintetizuje finalni odgovor na osnovu tri izvora:
- pronadjenih komada teksta iz vektorske baze (kontekst)
- pitanja korisnika
- istorije razgovora (pamcenje toka konverzacije)

LLM kombinuje sve informacije i generise jasan i prijatan odgovor. Svaka interakcija se dodaje u `chat_history` polje stanja, sto ostalim cvorovima u narednim pozivima omogucava da znaju sta je vec receno.

In [ ]:
def generate_node(state):
  context = "\n\n".join(state["documents"])
  question = state["question"]
  history = "\n".join(state.get("chat_history", []))

  prompt = f"""Ti si gotivan ljubazan asistent na sajtu Izdajem Iznajmljujem.
  Ovo je prethodna istorija razgovora sa ovim korisnikom:
  {history}
  Na osnovu KONTEKSTA, odgovori na PITANJE.
  KONTEKST: {context}
  PITANJE: {question}"""

  answer = llm.invoke(prompt).content

  new_interaction = [f"Korisnik: {question}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}

### `chat_node` - opsti razgovor (bypass RAG)

Kada router utvrdi da pitanje nije vezano za platformu (npr. matematika, casual small talk), sistem zaobilazi vektorsku pretragu i direktno poziva LLM. Ovo stedi tokene i smanjuje vreme odgovora za pitanja gde baza znanja ne moze da pomogne.

Cvor takodje ima pristup `chat_history` pa se ponasa kao pravi konverzacioni asistent - pamti sta je receno ranije u razgovoru i nadovezuje se prirodno.

In [ ]:
def chat_node(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history",[]))

  prompt = f"""Ti si gotivan ljubazan i dobar asistent na sajtu Izdajem Iznajmljujem.
  Ovo je prethodna istorija razgovora sa ovim korisnikom:
  {history}
  Korisnik ti kaže: '{question}'. Odgovori kratko i prijateljski."""

  answer = llm.invoke(prompt).content
  new_interaction = [f"Korisnik: {question}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}

### `escalate_node` - fallback mehanizam

Deterministicki cvor koji se aktivira kada `grade_node` proceni da pronadjeni komadi teksta ne sadrze odgovor na pitanje. Za razliku od ostalih cvorova, ovaj ne poziva LLM - vraca unapred definisanu poruku koja predlaze kontakt sa operaterom.

Ovo sprecava situaciju u kojoj bi model "izmislio" odgovor (hallucination) kada nema dovoljno informacija u bazi znanja, sto je posebno vazno jer se radi o pravnim i finansijskim informacijama platforme.

In [ ]:
def escalate_node(state):
  answer = "Nažalost, nemam tu informaciju u pravilniku. Da li želiš da te povežem sa operaterom?"
  new_interaction = [f"Korisnik: {state['question']}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}


### `router` - klasifikator namere

Prva tacka odlucivanja u grafu. Router koristi LLM da klasifikuje pitanje u jednu od dve kategorije: pitanje vezano za platformu (oglasi, krediti, depoziti, pravila) ili opsti razgovor.

Uzima se u obzir i poslednjih 5 poruka iz istorije razgovora - ako se korisnik nadovezuje na prethodnu temu vezanu za platformu, router ce to prepoznati i usmeriti pitanje na pretragu. Na ovaj nacin se sprecava nepotrebno pretrazivanje Chroma baze za small talk pitanja i smanjuju troskovi API poziva.

In [ ]:
def router(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history", [])[-5:]) # Gledamo samo zadnjih 5 poruka da uštedimo tokene

  prompt = f"""Da li se ovo pitanje odnosi na pravila, oglase, depozite, plaćanje, kredite ili funkcionisanje sajta za nekretnine?
  Uzmi u obzir i KONTEKST prethodnog razgovora (možda se korisnik nadovezuje na prethodnu temu).
  Prethodni razgovor: {history}
  Pitanje: {question}
  Odgovori ISKLJUČIVO sa DA ili NE."""

  decision = llm.invoke(prompt).content.strip().lower()

  if "da" in decision:
    return "retrieve_node"
  else:
    return "chat_node"

### `check_relevance` - uslovni prelaz na osnovu ocene

Staticka funkcija (ne koristi LLM) koja cita ocenu iz `grade_node` i usmerava tok grafa u jednom od dva smera:
- `is_relevant == "yes"` - prelaz na `generate_node` (ima dovoljno informacija za odgovor)
- `is_relevant == "no"` - prelaz na `escalate_node` (nema relevantnih podataka, aktivira se fallback)

In [ ]:
def check_relevance(state):
    if state["is_relevant"] == "yes":
        return "generate_node"
    else:
        return "escalate_node"

### Konstrukcija i kompajliranje grafa

Definisemo topologiju grafa: cvorovi se registruju, zatim se postavljaju ivice (edges) - obicne i uslovne. Uslovne ivice koriste `router` i `check_relevance` funkcije da odluce koji cvor dolazi sledeci u zavisnosti od sadrzaja stanja.

`MemorySaver` pamti stanje grafa izmedju poziva za isti `thread_id`, sto sistemu omogucava pamcenje istorije razgovora po korisniku.

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("retrieve_node", retrieve_node)
workflow.add_node("grade_node", grade_node)
workflow.add_node("generate_node", generate_node)
workflow.add_node("escalate_node", escalate_node)
workflow.add_node("chat_node", chat_node)

In [ ]:
workflow.add_conditional_edges(
    START,
    router,
    {
        "retrieve_node": "retrieve_node",
        "chat_node": "chat_node"
    }
)

In [ ]:
workflow.add_edge("retrieve_node", "grade_node")

In [ ]:
workflow.add_conditional_edges(
    "grade_node",
    check_relevance,
    {
        "generate_node": "generate_node",
        "escalate_node": "escalate_node"
    }
)

In [ ]:
workflow.add_edge("generate_node", END)
workflow.add_edge("escalate_node", END)
workflow.add_edge("chat_node", END)

In [ ]:
memory = MemorySaver()

In [ ]:
agent_app = workflow.compile(checkpointer= memory)
print("Graf je uspesno kompajliran na svim cvorovima")

## Faza 5: Testiranje

### Scenariji za sve grane grafa

Svaki poziv agenta mora imati jedinstven `thread_id` koji identifikuje korisnika. Na produkciji, ovaj ID dolazi sa backa (npr. ID korisnika iz JWT tokena). Isti `thread_id` tokom razgovora osigurava kontinuitet memorije - korisnik moze da postavi pratece pitanje a sistem ce razumeti kontekst.

Testiramo tri scenarija koji pokrivaju sve grane grafa:
1. Pitanje koje postoji u pravilniku (Router - Retrieve - Grade "yes" - Generate)
2. Pitanje o sajtu koje nije u pravilniku (Router - Retrieve - Grade "no" - Escalate)
3. Opsti razgovor koji nije vezan za platformu (Router - Chat)

In [ ]:
# config = {"configurable": {"thread_id": "USER_ID"}}

# print("\n--- PRVO PITANJE ---")
# q1 = "Koliki je rok za povraćaj kredita ako promocija nije iskorišćena?"
# print(f"KORISNIK: {q1}")
# res1 = agent_app.invoke({"question": q1}, config=config)
# print(f"BOT: {res1['answer']}")

# print("\n--- DRUGO PITANJE (Nadovezivanje - zahteva memoriju!) ---")
# # Obrati pažnju: Ne spominjemo "kredit" niti "povraćaj", samo kažemo "A na koji mejl to šaljem?"
# q2 = "A na koji mejl to šaljem?"
# print(f"KORISNIK: {q2}")
# # Koristimo ISTI config da bi znao da je to isti korisnik
# res2 = agent_app.invoke({"question": q2}, config=config)
# print(f"BOT: {res2['answer']}")

In [ ]:
config = {"configurable": {"thread_id": "test-user-1"}}

question_1 = "Da li se placa za oglas na vašem sajtu?"
print(f"KORISNIK: {question_1}")
result_1 = agent_app.invoke({"question": question_1}, config=config)
print(f"BOT: {result_1['answer']}\n")
print("-" * 50)

# Test 2: Pitanje za običan razgovor (Testiramo router!)
question_2 = "Da li znas kvadratnu jednacinu?"
print(f"\nKORISNIK: {question_2}")
result_2 = agent_app.invoke({"question": question_2}, config=config)
print(f"BOT: {result_2['answer']}")

In [ ]:
config = {"configurable": {"thread_id": "test-user-2"}}

print("\n--- TEST 1: Pitanje koje postoji u pravilniku (RAG -> GRADE -> GENERATE) ---")
q1 = "Koliko dana traje PRIORITY oglas i koliko košta?"
print(f"KORISNIK: {q1}")
print(f"BOT: {agent_app.invoke({'question': q1}, config=config)['answer']}")

print("\n--- TEST 2: Pitanje o sajtu, ali NE POSTOJI u pravilniku (RAG -> GRADE -> ESCALATE) ---")
q2 = "Kako da promenim temu sajta u crnu?"
print(f"KORISNIK: {q2}")
print(f"BOT: {agent_app.invoke({'question': q2}, config=config)['answer']}")

print("\n--- TEST 3: Običan razgovor (START -> CHAT) ---")
q3 = "Da li znas kvadratnu jednacinu?"
print(f"KORISNIK: {q3}")
print(f"BOT: {agent_app.invoke({'question': q3}, config=config)['answer']}")